# 01 — Data Exploration

**Project:** UREP 32-0210-250078 | Crack Classification

This notebook explores the dataset before training:
- Class distribution
- Sample images from each class
- Image size distribution
- Preprocessing pipeline visualization

In [ ]:
import sys
sys.path.insert(0, "..")

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter

import config
from src.preprocessing import (
    load_and_convert_rgb,
    resize_image,
    morphological_filter,
    apply_clahe,
    normalize_inception,
)
from src.dataset import collect_file_paths

sns.set_style("whitegrid")
print(f"Dataset directory: {config.DATA_DIR}")
print(f"Classes: {config.CLASS_NAMES}")

## 1. Class Distribution

In [ ]:
file_paths, labels = collect_file_paths(config.DATA_DIR)
label_counts = Counter(labels)

print(f"Total images: {len(file_paths)}")
for cls in config.CLASS_NAMES:
    print(f"  {cls}: {label_counts.get(cls, 0)}")

# Bar chart
fig, ax = plt.subplots(figsize=(8, 5))
classes = config.CLASS_NAMES
counts = [label_counts.get(c, 0) for c in classes]
colors = ["#e74c3c", "#3498db", "#f39c12", "#2ecc71"]
bars = ax.bar(classes, counts, color=colors, edgecolor="black", linewidth=0.5)

for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            str(count), ha="center", va="bottom", fontweight="bold")

ax.set_xlabel("Class")
ax.set_ylabel("Number of Images")
ax.set_title("Class Distribution")
plt.tight_layout()
plt.show()

## 2. Sample Images (4x4 Grid per Class)

In [ ]:
np.random.seed(config.RANDOM_SEED)

for class_name in config.CLASS_NAMES:
    class_files = [p for p, l in zip(file_paths, labels) if l == class_name]
    if len(class_files) == 0:
        print(f"No images found for class: {class_name}")
        continue

    sample_files = np.random.choice(class_files, size=min(16, len(class_files)), replace=False)

    fig, axes = plt.subplots(4, 4, figsize=(12, 12))
    fig.suptitle(f"Class: {class_name}", fontsize=16, fontweight="bold")

    for idx, ax in enumerate(axes.flat):
        if idx < len(sample_files):
            img = Image.open(sample_files[idx]).convert("RGB")
            ax.imshow(img)
            ax.set_title(f"{img.size[0]}x{img.size[1]}", fontsize=8)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

## 3. Image Size Distribution

In [ ]:
widths, heights = [], []
sample_size = min(500, len(file_paths))
sample_paths = np.random.choice(file_paths, size=sample_size, replace=False)

for path in sample_paths:
    with Image.open(path) as img:
        w, h = img.size
        widths.append(w)
        heights.append(h)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths, bins=30, color="#3498db", edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Width (px)")
axes[0].set_ylabel("Count")
axes[0].set_title("Image Width Distribution")

axes[1].hist(heights, bins=30, color="#e74c3c", edgecolor="black", alpha=0.7)
axes[1].set_xlabel("Height (px)")
axes[1].set_ylabel("Count")
axes[1].set_title("Image Height Distribution")

plt.tight_layout()
plt.show()

print(f"Width  — min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}")
print(f"Height — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}")

## 4. Preprocessing Pipeline Visualization

Shows each step of the 5-step pipeline on sample images.

In [ ]:
# Pick one sample from each class
for class_name in config.CLASS_NAMES:
    class_files = [p for p, l in zip(file_paths, labels) if l == class_name]
    if len(class_files) == 0:
        continue

    sample_path = np.random.choice(class_files)

    # Step 1: Load RGB
    img_pil = load_and_convert_rgb(sample_path)
    # Step 2: Resize
    img_resized = resize_image(img_pil)
    img_array = np.array(img_resized)
    # Step 3: Morphological filtering
    img_morph = morphological_filter(img_array)
    # Step 4: CLAHE
    img_clahe = apply_clahe(img_morph)
    # Step 5: Normalize (rescale for display)
    img_norm = normalize_inception(img_clahe)
    img_display = (img_norm + 1.0) / 2.0  # Rescale [-1,1] to [0,1] for display

    steps = [
        ("Original", np.array(img_pil)),
        ("Resized (299x299)", img_array),
        ("Morphological", img_morph),
        ("CLAHE", img_clahe),
        ("Normalized [-1,1]", img_display),
    ]

    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    fig.suptitle(f"Preprocessing Pipeline — {class_name}", fontsize=14, fontweight="bold")

    for ax, (title, img) in zip(axes, steps):
        ax.imshow(img)
        ax.set_title(title, fontsize=10)
        ax.axis("off")

    plt.tight_layout()
    plt.show()